# Stochastic Gradient Descent Experiment

This notebook covers part **H**: replacing full-batch gradient descent with
**stochastic gradient descent (SGD)** -- splitting the training data into
mini-batches and looping over epochs (Section 4.7 of the lecture notes) --
using the exact same `Optimizer` registry from parts E/F/G (`GradientDescent`
with `batch_size` set; see that class's docstring). It reuses the identical
degree-8 Runge dataset and OLS/Ridge split as `gradient_descent_experiment.ipynb`,
so results are directly comparable.

Three questions, matching the GitHub issue:

- How does the **mini-batch size** trade off against the **number of epochs**?
- How does the **learning-rate schedule** (constant vs. decaying) change SGD's
  behavior?
- Compared with full-batch gradient descent: how does SGD do on **final
  accuracy** relative to the OLS/Ridge closed-form solutions (parts A/B), and
  on **computational cost**?

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from fys_stk4155_p1.data.design_matrix import univariate_polynomial_design_matrix
from fys_stk4155_p1.data.runge import generate_runge_data
from fys_stk4155_p1.regression.cost import cost, hessian_max_eigenvalue
from fys_stk4155_p1.regression.gradient_descent import GradientDescent
from fys_stk4155_p1.regression.ridge import Ridge

## Setup: data, design matrix, scaling

Identical to `gradient_descent_experiment.ipynb`: degree 8, $n=200$, $\sigma=0.1$,
seed 42, non-intercept columns standardized on the training split,
`fit_intercept_column=True` throughout, and the same `lambdas = {"OLS": 0.0,
"Ridge": 0.01}` pair.

In [ ]:
def standardize(X_train, X_test):
    """Scale non-intercept columns (fit on train only); leave column 0 untouched."""
    scaler = StandardScaler()
    X_train_s = np.column_stack([X_train[:, :1], scaler.fit_transform(X_train[:, 1:])])
    X_test_s = np.column_stack([X_test[:, :1], scaler.transform(X_test[:, 1:])])
    return X_train_s, X_test_s


x, y = generate_runge_data(n=200, noise_std=0.1, seed=42)
X_full = univariate_polynomial_design_matrix(x=x, degree=8)

X_train, X_test, y_train, y_test = train_test_split(X_full, y, test_size=0.2, random_state=42)
X_train, X_test = standardize(X_train, X_test)

lambdas = {"OLS": 0.0, "Ridge": 0.01}

## Mini-batch size vs. computational cost

Epoch count alone isn't a fair x-axis across different mini-batch sizes: one
epoch at `batch_size=8` takes 20 optimizer updates on this training set (160
rows), while one epoch at `batch_size=160` takes a single update -- very
different amounts of work. `GradientDescent` tracks `cost_flops_` (Section 3
of the implementation plan; `regression.cost.gradient_flops`) alongside
`cost_history_` for exactly this reason: plotting cost against cumulative
FLOPs puts every batch size on the same, actually-comparable x-axis,
regardless of how many updates it took to get there.

We use OLS at degree 8 -- the harder, more revealing case from Part F, where
the design matrix's condition number (~54,000) makes plain gradient descent
and several optimizers struggle -- so any differences between mini-batch
sizes are easier to see than on the already-easy Ridge problem.

In [ ]:
gamma_max_ols = 2 / hessian_max_eigenvalue(X_train, lam=0.0, fit_intercept_column=True)
ols_closed_form = Ridge(lam=0.0, fit_intercept_column=True).fit(X_train, y_train)
ols_cost = cost(X_train, y_train, ols_closed_form.coef_, lam=0.0, fit_intercept_column=True)

batch_sizes = [8, 32, 128, X_train.shape[0]]
sweep_optimizers = ["plain", "adam"]
n_epochs_sweep = 500

fig, axes = plt.subplots(1, len(sweep_optimizers), figsize=(11, 4.5), sharey=True)
for ax, optimizer in zip(axes, sweep_optimizers, strict=True):
    for batch_size in batch_sizes:
        model = GradientDescent(
            learning_rate=0.9 * gamma_max_ols,
            lam=0.0,
            optimizer=optimizer,
            batch_size=batch_size,
            n_epochs=n_epochs_sweep,
            learning_rate_schedule="time_based",
            lr_decay=0.02,
            fit_intercept_column=True,
            random_state=0,
        ).fit(X_train, y_train)
        ax.plot(model.cost_flops_, model.cost_history_, label=f"batch={batch_size}")
        print(
            f"{optimizer:>6s}  batch={batch_size:4d}  "
            f"final cost / closed-form = {model.cost_history_[-1] / ols_cost:.3f}"
        )

    ax.axhline(ols_cost, color="black", ls=":", lw=1, label="closed-form")
    ax.set_yscale("log")
    ax.set_xlabel("cumulative FLOPs")
    ax.set_title(optimizer)
    ax.legend(fontsize="small")

axes[0].set_ylabel("Cost")
fig.suptitle("SGD: cost vs. computational cost, by mini-batch size (OLS, degree 8)")
fig.tight_layout()
plt.show()

At the same FLOP budget, smaller mini-batches reach a lower cost than larger
ones, for both optimizers -- `batch_size=8` beats `batch_size=160` (a single
mini-batch per epoch, equivalent to full-batch gradient descent with a
decaying learning rate) at every point along the curve. The mechanism: a
smaller batch means more, noisier updates per unit of compute; on this
ill-conditioned problem, more frequent (if noisier) progress along the slow
eigen-direction outweighs the extra noise, at least within the budget swept
here. Adam reaches a lower cost than plain at every batch size, consistent
with Part F.

## Learning-rate schedule

Plain gradient descent is only stable below $\gamma_{max} = 2/\lambda_{max}(H)$
(Part E). Push the learning rate above that boundary and combine it with
mini-batching -- does a decaying schedule recover stability that a constant
rate can't sustain?

In [ ]:
learning_rate_demo = 1.5 * gamma_max_ols  # 1.5x plain GD's own stability boundary
schedules_to_compare = [("constant", 0.0), ("time_based", 0.05), ("exponential", 0.05)]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ceiling = 1e3
with np.errstate(over="ignore", invalid="ignore"):
    for name, decay in schedules_to_compare:
        model = GradientDescent(
            learning_rate=learning_rate_demo,
            lam=0.0,
            optimizer="plain",
            batch_size=16,
            n_epochs=300,
            learning_rate_schedule=name,
            lr_decay=decay,
            fit_intercept_column=True,
            random_state=0,
        ).fit(X_train, y_train)
        # Some divergent runs produce huge-but-finite values before eventually
        # overflowing to inf/nan; nan_to_num alone only replaces the latter, so
        # clip explicitly to keep the y-axis from stretching to whatever huge
        # finite value happened to appear first.
        history = np.clip(
            np.nan_to_num(model.cost_history_, nan=ceiling, posinf=ceiling), None, ceiling
        )
        ax.plot(np.arange(1, model.n_iter_ + 1), history, label=f"{name} (decay={decay:g})")

ax.axhline(ols_cost, color="black", ls=":", lw=1, label="closed-form")
ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cost")
ax.set_title(r"Learning-rate schedule ($\gamma_0=1.5\times\gamma_{max}^{OLS}$, batch=16, plain)")
ax.legend(fontsize="small")
fig.tight_layout()
plt.show()

At $\gamma_0 = 1.5\gamma_{max}$, the **constant** schedule diverges outright
(plain gradient descent has no mechanism to recover once a step overshoots
this badly, mini-batching or not). Both decaying schedules stabilize
training, but not equally well: **time-based** decay (`lr_decay=0.05`)
reaches a noticeably lower final cost than **exponential** decay at the same
nominal decay rate here -- decaying *is* necessary to survive an
otherwise-unstable learning rate, but the specific schedule and its rate
still need tuning, not just "any decay will do."

## With vs. without SGD: accuracy and computational cost

Head-to-head at a matched FLOP budget (500 full-batch iterations, or 500 SGD
epochs at `batch_size=8` -- both processing the same total amount of data,
per the mini-batch-size section above): final cost relative to the
closed-form solution, total FLOPs, and **wall-clock time**. FLOPs and
wall-clock time can disagree -- Python-loop overhead per mini-batch update is
a real cost this implementation pays that a raw FLOP count doesn't capture.

In [ ]:
configs = [
    ("full-batch, plain", {"optimizer": "plain", "max_iter": 500, "tol": 0.0}),
    ("full-batch, adam", {"optimizer": "adam", "max_iter": 500, "tol": 0.0}),
    (
        "SGD, adam, batch=8",
        {
            "optimizer": "adam",
            "batch_size": 8,
            "n_epochs": 500,
            "learning_rate_schedule": "time_based",
            "lr_decay": 0.02,
            "random_state": 0,
        },
    ),
]

header = (
    f"{'config':>20s} {'penalty':>8s} {'cost/closed-form':>17s} {'FLOPs':>12s} {'time (ms)':>10s}"
)
print(header)
for label, lam in lambdas.items():
    gamma_max = 2 / hessian_max_eigenvalue(X_train, lam=lam, fit_intercept_column=True)
    closed_form = Ridge(lam=lam, fit_intercept_column=True).fit(X_train, y_train)
    closed_form_cost = cost(X_train, y_train, closed_form.coef_, lam, fit_intercept_column=True)

    for name, kwargs in configs:
        start = time.perf_counter()
        model = GradientDescent(
            learning_rate=0.9 * gamma_max, lam=lam, fit_intercept_column=True, **kwargs
        ).fit(X_train, y_train)
        elapsed_ms = (time.perf_counter() - start) * 1000
        ratio = model.cost_history_[-1] / closed_form_cost
        print(
            f"{name:>20s} {label:>8s} {ratio:>17.3f} "
            f"{model.cost_flops_[-1]:>12.3e} {elapsed_ms:>10.2f}"
        )

## Discussion

- **On Ridge, everything reaches the closed-form solution essentially
  exactly** -- full-batch plain, full-batch Adam, and mini-batch SGD alike,
  matching Part F's finding that Ridge's condition number is low enough that
  optimizer choice barely matters once the problem itself is well-posed. The
  only real differentiator left on Ridge is computational cost, not accuracy.
- **On OLS, full-batch Adam is the best performer at this FLOP budget** --
  ahead of full-batch plain, and slightly ahead of mini-batch SGD despite the
  mini-batch-size section above showing smaller batches winning *within* the
  SGD family. The batch-size comparison there held the optimizer and FLOP
  budget fixed and only varied `batch_size`; here, full-batch Adam still
  edges out `batch_size=8` Adam at the same total FLOPs, since it's not just
  update *count* that matters but how much curvature information Adam's
  moment estimates capture per update, and a full-batch gradient is a less
  noisy signal than a batch of 8.
- **Wall-clock time tells a different story than FLOPs.** Despite matching
  FLOP budgets, `batch_size=8` SGD takes roughly an order of magnitude longer
  in wall-clock time than full-batch gradient descent, because it performs
  20x as many `optimizer.step` calls -- each individually cheap in raw
  arithmetic, but Python-loop and per-call overhead dominates at this problem
  size (160 training rows, 9 features) rather than the matrix-vector products
  `gradient_flops` counts. In a NumPy/pure-Python implementation on a small,
  fully-in-memory dataset like this one, full-batch gradient descent is
  simply the faster choice on both axes measured here.
- **This doesn't mean SGD is pointless -- it means its advantage is
  conditional.** SGD's real motivation is datasets too large to hold a
  full-batch gradient in memory or compute in one vectorized step, where
  "more, cheaper updates" genuinely reduces wall-clock time rather than just
  FLOP count. At this project's scale, that condition doesn't hold, so the
  honest conclusion is: mini-batching can still reach a lower cost *per FLOP
  spent* than full-batch (the first section above), but that FLOP-efficiency
  doesn't translate into a wall-clock win here, and a decaying learning-rate
  schedule is necessary, not optional, once the learning rate is pushed past
  what plain gradient descent can otherwise sustain.